# 🧠 Fake News Detection — LSTM Model Training

> **Bidirectional LSTM** · Binary Classification · NLP Pipeline  
> Dataset: [Kaggle — Fake and Real News](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)

### Pipeline
`Load Data` → `Clean Text` → `Tokenize & Pad` → `Train BiLSTM` → `Evaluate` → `Save Model` → `Streamlit App`

In [ ]:
# ─── Install required packages (run once) ───
# !pip install tensorflow pandas numpy scikit-learn matplotlib seaborn nltk streamlit

import os
import sys
import re
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─── TensorFlow + GPU Setup ───
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"✔ GPU detected: {[g.name for g in gpus]}")
    print(f"✔ Mixed-precision: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("⚠ No GPU — training on CPU (slower)")

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout,
                                     Bidirectional, SpatialDropout1D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words("english"))

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

## 1 · Load Raw Datasets

Load **Fake.csv** and **True.csv** from `data/raw/`, label them, and merge into a single DataFrame.

In [ ]:
# Adjust path — notebook lives in notebooks/, data in data/raw/
DATA_DIR = os.path.join("..", "data", "raw")

df_fake = pd.read_csv(os.path.join(DATA_DIR, "Fake.csv"))
df_true = pd.read_csv(os.path.join(DATA_DIR, "True.csv"))

df_fake["label"] = 0   # 0 → Fake
df_true["label"] = 1   # 1 → Real

df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Fake articles : {len(df_fake):,}")
print(f"Real articles : {len(df_true):,}")
print(f"Combined      : {len(df):,}")
df.head()

## 2 · Explore and Understand the Data

Check class balance, missing values, and text-length distributions.

In [ ]:
# ─── Basic info ───
print("Columns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nLabel distribution:\n{df['label'].value_counts().rename({0:'Fake', 1:'Real'})}")

# ─── Class distribution bar chart ───
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["label"].value_counts().rename({0: "Fake", 1: "Real"}).plot.bar(
    ax=axes[0], color=["#e74c3c", "#2ecc71"], edgecolor="black"
)
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# ─── Text length distribution ───
df["text_len"] = df["text"].astype(str).apply(len)
df[df["label"] == 0]["text_len"].hist(ax=axes[1], bins=50, alpha=0.6, color="#e74c3c", label="Fake")
df[df["label"] == 1]["text_len"].hist(ax=axes[1], bins=50, alpha=0.6, color="#2ecc71", label="Real")
axes[1].set_title("Text Length Distribution")
axes[1].set_xlabel("Character count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3 · Data Cleaning and Preprocessing

- Combine **title** + **text** into one feature  
- Lowercase, strip URLs, HTML tags, punctuation, digits  
- Remove English stopwords

In [ ]:
def clean_text(text: str) -> str:
    """Full cleaning pipeline for a single string."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # URLs
    text = re.sub(r"<.*?>", "", text)                  # HTML tags
    text = re.sub(r"[^a-z\s]", "", text)               # keep only letters
    tokens = text.split()
    tokens = [w for w in tokens if w not in STOP_WORDS]
    return " ".join(tokens)

# Combine title + text, then clean
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna(""))
df["clean_text"] = df["content"].apply(clean_text)

# Drop columns we no longer need
df.drop(columns=["title", "text", "subject", "date", "content", "text_len"],
        inplace=True, errors="ignore")

print(f"Cleaned dataset shape: {df.shape}")
df[["clean_text", "label"]].head()

## 4 · Text Tokenization and Sequence Padding

Convert cleaned text into integer sequences using Keras `Tokenizer`, then pad to a uniform length.

In [ ]:
MAX_WORDS = 10_000    # vocabulary cap
MAX_LEN   = 300       # max sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])

sequences = tokenizer.texts_to_sequences(df["clean_text"])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
y = df["label"].values

print(f"Vocabulary size  : {min(MAX_WORDS, len(tokenizer.word_index)+1)}")
print(f"Padded X shape   : {X.shape}")
print(f"Labels y shape   : {y.shape}")
print(f"\nSample padded sequence (first 20 tokens): {X[0][:20]}")

## 5 · Train-Test Split

80/20 split with stratified sampling to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")

## 6 · Build the LSTM Model Architecture

**Embedding → SpatialDropout1D → Bidirectional LSTM → Dense → Dropout → Sigmoid**

In [ ]:
EMBEDDING_DIM = 128
LSTM_UNITS    = 64

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

## 7 · Train the LSTM Model

Training for up to **10 epochs** with **EarlyStopping** (patience=3) and **ModelCheckpoint** to save the best weights.

In [ ]:
MODEL_DIR  = os.path.join("..", "models")
MODEL_PATH = os.path.join(MODEL_DIR, "fake_news_lstm.h5")
os.makedirs(MODEL_DIR, exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

## 8 · Evaluate Model Performance

Test-set accuracy, classification report, and confusion matrix.

In [ ]:
# ─── Evaluate on test set ───
loss, acc = model.evaluate(X_test, y_test, verbose=0)

# ─── Predictions ───
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# ─── Classification report ───
print("=" * 55)
print(f"  Test Loss     : {loss:.4f}")
print(f"  Test Accuracy : {acc:.4f}")
print("=" * 55)
print()
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

# ─── Confusion matrix + accuracy/loss curves ───
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1 — Confusion Matrix (styled)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="RdYlGn",
            xticklabels=["Fake", "Real"],
            yticklabels=["Fake", "Real"],
            linewidths=1, linecolor="white",
            annot_kws={"size": 16, "weight": "bold"},
            ax=axes[0])
axes[0].set_xlabel("Predicted", fontsize=12)
axes[0].set_ylabel("Actual", fontsize=12)
axes[0].set_title("Confusion Matrix", fontsize=14, fontweight="bold")

# 2 — Accuracy curve
axes[1].plot(history.history["accuracy"],     "o-", label="Train", linewidth=2)
axes[1].plot(history.history["val_accuracy"],  "s--", label="Validation", linewidth=2)
axes[1].set_title("Accuracy per Epoch", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# 3 — Loss curve
axes[2].plot(history.history["loss"],     "o-", label="Train", linewidth=2, color="#e74c3c")
axes[2].plot(history.history["val_loss"],  "s--", label="Validation", linewidth=2, color="#3498db")
axes[2].set_title("Loss per Epoch", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✔ Plots saved to {MODEL_DIR}/training_curves.png")

## 9 · Save Trained Model and Tokenizer

Persist the model (`.h5`) and tokenizer (`.pkl`) so they can be loaded by the Streamlit app and the `predict.py` module.

In [ ]:
# Save the trained model
model.save(MODEL_PATH)
print(f"✔ Model saved to {MODEL_PATH}")

# Save tokenizer
TOKENIZER_PATH = os.path.join(MODEL_DIR, "tokenizer.pkl")
with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"✔ Tokenizer saved to {TOKENIZER_PATH}")

# Save max_len config
CONFIG_PATH = os.path.join(MODEL_DIR, "config.pkl")
with open(CONFIG_PATH, "wb") as f:
    pickle.dump({"max_len": MAX_LEN, "max_words": MAX_WORDS}, f)
print(f"✔ Config saved to {CONFIG_PATH}")

# Verify
print(f"\nFiles in {MODEL_DIR}/:")
for fname in os.listdir(MODEL_DIR):
    fsize = os.path.getsize(os.path.join(MODEL_DIR, fname))
    print(f"  {fname:30s} {fsize/1024:.1f} KB")

## 10 · Test Prediction on Sample Text

Quick sanity check — predict on sample fake and real news snippets.

In [ ]:
def predict_single(text: str):
    """Quick prediction helper using the model already in memory."""
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Real" if prob >= 0.5 else "Fake"
    confidence = prob if prob >= 0.5 else 1 - prob
    return label, confidence

# ─── Sample predictions ───
samples = [
    "Breaking: Scientists confirm the earth is flat and NASA has been lying for decades!",
    "The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, "
    "in line with market expectations, citing continued strength in the labor market.",
]

for i, txt in enumerate(samples, 1):
    label, conf = predict_single(txt)
    print(f"Sample {i}: {label} (confidence: {conf:.2%})")
    print(f"  Text: {txt[:100]}...\n")

## Conclusion

- The **Bidirectional LSTM** model achieves strong accuracy on the Fake/Real news classification task.  
- The trained model and tokenizer are saved under `models/` and can be loaded by:
  - `src/predict.py` — CLI predictions  
  - `app/streamlit_app.py` — interactive web UI  
- **Next step:** launch the Streamlit app with `streamlit run app/streamlit_app.py`